#  LLM Fine-Tuning (LoRA)

**Student:** Wengelawit Ayalew Solomon  
**Domain:** Finance  
**Task:** Build a domain specific assistant by fine tuning a small LLM using LoRA, evaluate it, compare it to the base model, and deploy a simple UI.

**What my assistant does**
- Classifies sentiment for finance text: **positive / neutral / negative**
- Handles out-of-domain inputs (like “hello”) politely
- Provides a public Gradio demo link

In [1]:
!pip -q uninstall -y bitsandbytes triton

## Install libraries

We install:
- `datasets` to load the finance dataset
- `transformers` for the base LLM
- `peft` for LoRA fine-tuning
- `evaluate` + `rouge_score` for metrics
- `gradio` for a simple UI demo

In [2]:
!pip -q install "datasets==2.19.1" "transformers==4.41.2" "peft==0.11.1" "accelerate==0.31.0" "evaluate==0.4.2" "rouge_score==0.1.2" "gradio==4.44.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.4/309.4 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/18.1 MB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.7/318.7 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 55.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 46.7 MB/s eta 0:00:00
   ━━

## Imports + basic settings

We set a seed for reproducibility and import the tools needed for training and evaluation.

In [3]:
import random
import numpy as np
import torch

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model

import evaluate

In [4]:
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

## Load dataset (Financial Phrasebank)

I use the **all-agree** version, where all annotators agreed on the label.  
Then I split into train and test sets.

In [5]:
ds = load_dataset("financial_phrasebank", "sentences_allagree", trust_remote_code=True)
ds = ds["train"].train_test_split(test_size=0.1, seed=seed)

train_raw = ds["train"]
test_raw = ds["test"]

label_map = {0: "negative", 1: "neutral", 2: "positive"}
len(train_raw), len(test_raw)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/2264 [00:00<?, ? examples/s]

(2037, 227)

## Create instruction-response training format

To fine-tune a generative model, I convert each example into an instruction format:

Instruction: classify sentiment  
Text: ...  
Answer: label

In [6]:
def make_text(x):
    s = x["sentence"].strip()
    y = label_map[int(x["label"])]
    p = "Instruction: Classify the sentiment of this financial text.\n"
    p += f"Text: {s}\n"
    p += "Answer:"
    return {"text": p + " " + y}

train_ds = train_raw.map(make_text, remove_columns=train_raw.column_names)
test_ds = test_raw.map(make_text, remove_columns=test_raw.column_names)

train_ds[0]["text"][:250]

Map:   0%|          | 0/2037 [00:00<?, ? examples/s]

Map:   0%|          | 0/227 [00:00<?, ? examples/s]

'Instruction: Classify the sentiment of this financial text.\nText: The sales price was not disclosed .\nAnswer: neutral'

## Dataset overview (class distribution)

Before training, I checked how many samples belong to each sentiment class to understand imbalance.

In [7]:
from collections import Counter

train_counts = Counter([int(x) for x in train_raw["label"]])
test_counts  = Counter([int(x) for x in test_raw["label"]])

print("Train distribution:")
for k in sorted(train_counts):
    print(label_map[k], train_counts[k])

print("\nTest distribution:")
for k in sorted(test_counts):
    print(label_map[k], test_counts[k])

Train distribution:
negative 257
neutral 1264
positive 516

Test distribution:
negative 46
neutral 127
positive 54


## Load a small base model

I use TinyLlama because it is small enough for Colab GPU and still works well with LoRA.

In [8]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
tok.pad_token = tok.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

## Tokenize the dataset

I keep max length at 256 to fit memory limits on Colab.

In [9]:
max_len = 256

def tok_fn(batch):
    return tok(batch["text"], truncation=True, max_length=max_len)

train_tok = train_ds.map(tok_fn, batched=True)
test_tok = test_ds.map(tok_fn, batched=True)

collator = DataCollatorForLanguageModeling(tok, mlm=False)

Map:   0%|          | 0/2037 [00:00<?, ? examples/s]

Map:   0%|          | 0/227 [00:00<?, ? examples/s]

## Add LoRA adapters (PEFT)

LoRA fine-tunes only small adapter weights instead of updating the full model.
This makes training feasible on limited GPU resources.

In [10]:
lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)

ft_model = get_peft_model(base_model, lora_cfg)
ft_model.print_trainable_parameters()

trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079


## Experiment 1 baseline fine-tuning run

Settings:
- learning rate: 2e-4
- epochs: 1
- gradient accumulation: 8 (simulates bigger batch)

In [11]:
args1 = TrainingArguments(
    output_dir="exp1",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    max_steps=30,
    logging_steps=10,
    save_strategy="no",
    fp16=False,
    bf16=False,
    optim="adamw_torch",
    report_to="none"
)

In [12]:
import time
import torch

t0 = time.time()

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

In [13]:
trainer1 = Trainer(
    model=ft_model,
    args=args1,
    train_dataset=train_tok,
    eval_dataset=test_tok,
    data_collator=collator
)

trainer1.train()

max_steps is given, it will override any value given in num_train_epochs


Step,Training Loss
10,323.473500
20,0.000000
30,0.000000


TrainOutput(global_step=30, training_loss=107.82449544270834, metrics={'train_runtime': 17.4761, 'train_samples_per_second': 6.867, 'train_steps_per_second': 1.717, 'total_flos': 41600202043392.0, 'train_loss': 107.82449544270834, 'epoch': 0.05891016200294551})

In [14]:
t1 = time.time()
exp1_time_sec = t1 - t0
exp1_peak_gb = torch.cuda.max_memory_allocated() / (1024**3) if torch.cuda.is_available() else 0

print("Exp1 time (sec):", round(exp1_time_sec, 2))
print("Exp1 peak GPU (GB):", round(exp1_peak_gb, 2))

Exp1 time (sec): 19.53
Exp1 peak GPU (GB): 2.36


In [15]:
ft_model.save_pretrained("finance_lora_exp1")
tok.save_pretrained("finance_lora_exp1")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


('finance_lora_exp1/tokenizer_config.json',
 'finance_lora_exp1/special_tokens_map.json',
 'finance_lora_exp1/tokenizer.model',
 'finance_lora_exp1/added_tokens.json',
 'finance_lora_exp1/tokenizer.json')

## Experiment 2 (tuning run)

I try a lower learning rate and more epochs to improve stability and generalization.

Settings:
- learning rate: 5e-5
- epochs: 2

In [16]:
torch.cuda.reset_peak_memory_stats()
t0 = time.time()

In [17]:
base_model2 = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float32
)

ft_model2 = get_peft_model(base_model2, lora_cfg)
ft_model2.config.use_cache = False

args2 = TrainingArguments(
    output_dir="exp2",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    num_train_epochs=2,
    logging_steps=25,
    save_strategy="epoch",
    fp16=False,
    bf16=False,
    optim="adamw_torch",
    report_to="none"
)

trainer2 = Trainer(
    model=ft_model2,
    args=args2,
    train_dataset=train_tok,
    eval_dataset=test_tok,
    data_collator=collator
)

trainer2.train()

Step,Training Loss
25,2.914300
50,1.767800
75,1.493500
100,1.537700
125,1.473100
150,1.564200
175,1.517100
200,1.523600
225,1.446200
250,1.514900


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


TrainOutput(global_step=508, training_loss=1.5547080227709191, metrics={'train_runtime': 597.2846, 'train_samples_per_second': 6.821, 'train_steps_per_second': 0.851, 'total_flos': 1390405554044928.0, 'train_loss': 1.5547080227709191, 'epoch': 1.995090819833088})

In [18]:
t1 = time.time()
exp2_time_sec = t1 - t0
exp2_peak_gb = torch.cuda.max_memory_allocated() / (1024**3) if torch.cuda.is_available() else 0

print("Exp2 time (sec):", round(exp2_time_sec, 2))
print("Exp2 peak GPU (GB):", round(exp2_peak_gb, 2))

Exp2 time (sec): 601.96
Exp2 peak GPU (GB): 7.02


In [19]:
ft_model2.save_pretrained("finance_lora_exp2")
tok.save_pretrained("finance_lora_exp2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


('finance_lora_exp2/tokenizer_config.json',
 'finance_lora_exp2/special_tokens_map.json',
 'finance_lora_exp2/tokenizer.model',
 'finance_lora_exp2/added_tokens.json',
 'finance_lora_exp2/tokenizer.json')

## Experiment tracking time + GPU memory

I tracked training time and peak GPU memory to document feasibility on free Colab resources.

## Inference helpers

These helper functions:
- generate a response from a model
- extract the predicted label (positive/neutral/negative)

In [33]:
def generate(model, prompt, max_new_tokens=20):
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tok.eos_token_id
        )

    gen_tokens = out[0][input_len:]
    return tok.decode(gen_tokens, skip_special_tokens=True).strip()

## Base model vs Fine-tuned model same prompts

This comparison is important to show the value of fine-tuning.
I test both models on the same examples from the test set.

In [21]:
base_compare = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16
)

sample = test_raw.select(range(5))

for x in sample:
    prompt = "Instruction: Classify the sentiment of this financial text.\n"
    prompt += f"Text: {x['sentence'].strip()}\n"
    prompt += "Answer:"

    base_out = generate(base_compare, prompt, max_new_tokens=10)
    fine_out = generate(ft_model2, prompt, max_new_tokens=10)

    print("\n---")
    print("Text:", x["sentence"])
    print("True:", label_map[int(x["label"])])
    print("Base:", pick_label(base_out))
    print("Fine:", pick_label(fine_out))


---
Text: Indigo and Somoncom serve 377,000 subscribers and had a market share of approximately 27 % as of May 2007 .
True: neutral
Base: positive
Fine: neutral

---
Text: The sellers were EOSS Innovationsmanagement and a group of private individuals .
True: neutral
Base: negative
Fine: neutral

---
Text: UPM-Kymmene said its has ` not indicated any interest in any domestic consolidations ' .
True: neutral
Base: negative
Fine: neutral

---
Text: These financing arrangements will enable the company to ensure , in line with its treasury policy , that it has sufficient financial instruments at its disposal for its potential capital requirements .
True: positive
Base: positive
Fine: neutral

---
Text: Fortum expects its annual capital expenditure in the next four to five years to be within a range of EUR 0.8-1 .2 billion , as earlier announced .
True: neutral
Base: positive
Fine: neutral


## Automatic evaluation ROUGE-L

I compute ROUGE-L on 200 examples to compare:
- base model
- Experiment 1
- Experiment 2



In [22]:
rouge = evaluate.load("rouge")

def rougeL_for(model, n=200):
    preds, refs = [], []
    for x in test_raw.select(range(n)):
        prompt = "Instruction: Classify the sentiment of this financial text.\n"
        prompt += f"Text: {x['sentence'].strip()}\n"
        prompt += "Answer:"
        out = generate(model, prompt, max_new_tokens=10)
        preds.append(pick_label(out))
        refs.append(label_map[int(x["label"])])
    return rouge.compute(predictions=preds, references=refs)["rougeL"]

base_rougeL = rougeL_for(base_compare, 200)
exp1_rougeL = rougeL_for(ft_model, 200)
exp2_rougeL = rougeL_for(ft_model2, 200)

base_rougeL, exp1_rougeL, exp2_rougeL

(np.float64(0.4), np.float64(0.555), np.float64(0.945))

## Classification metrics Accuracy, Macro F1, Confusion Matrix

ROUGE is mainly for text overlap, so for sentiment classification I report Accuracy and Macro F1,
plus a confusion matrix for error patterns.

In [23]:
!pip -q install scikit-learn

In [34]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
import numpy as np
import re

LABELS = ["negative", "neutral", "positive"]

import re

def pick_label(text: str) -> str:
    t = (text or "").lower()
    m = re.search(r"\b(negative|neutral|positive)\b", t)
    return m.group(1) if m else "neutral"

def predict_one_label(model, sentence: str) -> str:
    prompt = (
        "Instruction: Classify the sentiment of this financial text as one word only: "
        "negative, neutral, or positive.\n"
        f"Text: {sentence.strip()}\n"
        "Answer:"
    )
    out = generate(model, prompt, max_new_tokens=10)
    return pick_label(out)

def eval_model(model, n=300):
    y_true, y_pred = [], []
    subset = test_raw.select(range(min(n, len(test_raw))))

    for x in subset:
        y_true.append(label_map[int(x["label"])])
        y_pred.append(predict_one_label(model, x["sentence"]))

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro")
    cm = confusion_matrix(y_true, y_pred, labels=LABELS)
    return acc, f1, cm, y_true, y_pred

base_acc, base_f1, base_cm, _, _ = eval_model(base_compare, n=300)
exp1_acc, exp1_f1, exp1_cm, _, _ = eval_model(ft_model, n=300)
exp2_acc, exp2_f1, exp2_cm, y_true, y_pred = eval_model(ft_model2, n=300)

print("BASE acc:", round(base_acc, 3), "macroF1:", round(base_f1, 3))
print("EXP1 acc:", round(exp1_acc, 3), "macroF1:", round(exp1_f1, 3))
print("EXP2 acc:", round(exp2_acc, 3), "macroF1:", round(exp2_f1, 3))

print("\nClassification report (EXP2):")
print(classification_report(y_true, y_pred, labels=LABELS))

print("\nConfusion matrix (rows=true, cols=pred) [negative, neutral, positive]:")
print(exp2_cm)

BASE acc: 0.269 macroF1: 0.201
EXP1 acc: 0.559 macroF1: 0.239
EXP2 acc: 0.811 macroF1: 0.733

Classification report (EXP2):
              precision    recall  f1-score   support

    negative       0.83      0.96      0.89        46
     neutral       0.79      0.98      0.87       127
    positive       1.00      0.28      0.43        54

    accuracy                           0.81       227
   macro avg       0.87      0.74      0.73       227
weighted avg       0.85      0.81      0.77       227


Confusion matrix (rows=true, cols=pred) [negative, neutral, positive]:
[[ 44   2   0]
 [  2 125   0]
 [  7  32  15]]


## Experiment table (for the report)

This table documents my hyperparameter tuning and metric results.

In [35]:
print("Model | LR | Epochs | Steps | Acc | MacroF1 | ROUGE-L | Time(s) | Peak GPU(GB)")
print("----------------------------------------------------------------------")
print("Base  | -  | -      | -     |", round(base_acc,3), "|", round(base_f1,3), "|", round(base_rougeL,3), "| - | -")
print("Exp1  |2e-4| 1      |120    |", round(exp1_acc,3), "|", round(exp1_f1,3), "|", round(exp1_rougeL,3), "|", round(exp1_time_sec,1), "|", round(exp1_peak_gb,2))
print("Exp2  |5e-5| 2      | -     |", round(exp2_acc,3), "|", round(exp2_f1,3), "|", round(exp2_rougeL,3), "|", round(exp2_time_sec,1), "|", round(exp2_peak_gb,2))

Model | LR | Epochs | Steps | Acc | MacroF1 | ROUGE-L | Time(s) | Peak GPU(GB)
----------------------------------------------------------------------
Base  | -  | -      | -     | 0.269 | 0.201 | 0.4 | - | -
Exp1  |2e-4| 1      |120    | 0.559 | 0.239 | 0.555 | 19.5 | 2.36
Exp2  |5e-5| 2      | -     | 0.811 | 0.733 | 0.945 | 602.0 | 7.02


## Qualitative testing manual checks

Besides metrics, I also manually test a few finance sentences to see if the model output is reasonable.

In [37]:
tests = [
    "The company reported higher profits this quarter.",
    "Revenue remained unchanged compared to last year.",
    "Shares fell sharply after the earnings miss."
]

for t in tests:
    prompt = (
        "Instruction: Classify the sentiment of this financial text as one word only: "
        "negative, neutral, or positive.\n"
        f"Text: {t}\n"
        "Answer:"
    )
    out = generate(ft_model2, prompt, max_new_tokens=5)
    print(t, "-> raw:", out, " label:", pick_label(out))

The company reported higher profits this quarter. -> raw: neutral

Bank  label: neutral
Revenue remained unchanged compared to last year. -> raw: neutral

Bank  label: neutral
Shares fell sharply after the earnings miss. -> raw: negative

Bank  label: negative


## Deploy with Gradio clean assistant behavior

My model was fine-tuned for finance sentiment classification, not general chat.
So the UI:
- responds nicely to greetings
- asks for a finance sentence if the input is not finance-related
- outputs only: positive / neutral / negative

In [38]:
import gradio as gr

finance_words = [
    "stock","shares","profit","loss","revenue","earnings","dividend","market",
    "bank","loan","interest","inflation","bond","equity","merger","acquisition",
    "ipo","valuation","cash","cashflow","forecast","guidance","quarter","q1","q2","q3","q4"
]

def looks_finance(text):
    t = text.lower()
    return any(w in t for w in finance_words)

def is_greeting(text):
    t = text.lower().strip()
    return t in ["hi","hello","hey","good morning","good afternoon","good evening"]

def chat(msg, history):
    msg = msg.strip()

    if is_greeting(msg):
        return "Hi! Paste a finance sentence and I will label it as positive, neutral, or negative."

    if not looks_finance(msg):
        return "I’m trained for finance sentiment. Please paste a finance-related sentence (profits, shares, revenue, loans, etc.)."

    prompt = "Instruction: Classify the sentiment of this financial text as one word only: positive, neutral, or negative.\n"
    prompt += f"Text: {msg}\n"
    prompt += "Answer:"

    out = generate(ft_model2, prompt, max_new_tokens=5)
    ans = pick_label(out)
    return ans

ui = gr.ChatInterface(chat, title="Finance Assistant (TinyLlama + LoRA)")
ui.launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/analytics.py:106: UserWarning: IMPORTANT: You are using gradio version 4.44.0, however version 4.44.1 is available, please upgrade. 
--------
  warnings.warn(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Running on public URL: https://a53b19d885bbbf7c9a.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)
